# Diabetes Risk Prediction — Full Analysis Notebook

This notebook trains a **Random Forest classifier** on the BRFSS 2015 diabetes health-indicators dataset and saves the model artifacts used by the Streamlit app.

## Workflow
1. Load the data
2. Inspect shape, missing values, duplicates, and dtypes
3. Clean — drop duplicates, remove missing/invalid rows, fix dtypes
4. Create EDA plots
5. Run correlation analysis
6. Split into train/test data
7. Train Random Forest
8. Evaluate the model
9. Plot feature importance
10. Save model artifacts


## 0. Setup

In [ ]:
# If any package is missing, install project dependencies from the project root:
# !pip install -r ../requirements.txt

# Import file/path tools used to build robust project-relative paths.
import json
from pathlib import Path

# Import data-analysis and plotting libraries.
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine-learning tools for splitting, training, and evaluating the model.
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)

# Set one visual style so all notebook plots look consistent.
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.dpi"] = 120
TEAL, AMBER = "#0F766E", "#F59E0B"

# Resolve the project root whether Jupyter starts inside notebooks/ or at project root.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

# Store the dataset path in one variable so every later cell reads the same CSV.
DATA_PATH = PROJECT_ROOT / "data" / "diabetes_binary_5050split_health_indicators_BRFSS2015.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

# Create output folders before saving model files and plots.
MODELS_DIR = PROJECT_ROOT / "models"
PLOTS_DIR = PROJECT_ROOT / "plots"
MODELS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data path   :", DATA_PATH)


## 1. Load the data

In [ ]:
# Read the BRFSS CSV file into a dataframe.
df = pd.read_csv(DATA_PATH)

# Display the initial size and first rows to verify the file loaded correctly.
print("Raw shape:", df.shape)
df.head()


## 2. Inspect

In [ ]:
# Show column names, non-null counts, and current dtypes.
df.info()


In [ ]:
# Summarise the numeric columns to check ranges and possible abnormal values.
df.describe().round(2)


In [ ]:
# Count missing cells and duplicate rows before cleaning.
print("Missing cells per column:")
print(df.isna().sum())
print("
Total missing cells:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

# Check whether the target classes are balanced.
print("
Class balance:")
print(df["Diabetes_binary"].value_counts(normalize=True).round(3))


## 3. Clean — drop duplicates, remove missing/invalid rows, fix dtypes

In [ ]:
# Define the exact columns expected by the model and keep them in training order.
EXPECTED_COLUMNS = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "BMI", "Smoker",
    "Stroke", "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth", "MentHlth",
    "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

# Define valid BRFSS coding ranges to catch edited or accidentally added bad rows.
VALUE_RANGES = {
    "Diabetes_binary": (0, 1), "HighBP": (0, 1), "HighChol": (0, 1),
    "CholCheck": (0, 1), "BMI": (12, 98), "Smoker": (0, 1), "Stroke": (0, 1),
    "HeartDiseaseorAttack": (0, 1), "PhysActivity": (0, 1), "Fruits": (0, 1),
    "Veggies": (0, 1), "HvyAlcoholConsump": (0, 1), "AnyHealthcare": (0, 1),
    "NoDocbcCost": (0, 1), "GenHlth": (1, 5), "MentHlth": (0, 30),
    "PhysHlth": (0, 30), "DiffWalk": (0, 1), "Sex": (0, 1), "Age": (1, 13),
    "Education": (1, 6), "Income": (1, 8),
}

# Validate that the CSV has all required columns before modeling.
missing_columns = [col for col in EXPECTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# Keep only the expected columns; this removes any accidental extra columns.
df = df[EXPECTED_COLUMNS].copy()

# Convert all columns to numeric; any edited text values become NaN and are removed below.
df = df.apply(pd.to_numeric, errors="coerce")

# Count cleaning issues before removing rows.
duplicates_before = int(df.duplicated().sum())
missing_rows_before = int(df.isna().any(axis=1).sum())
print("Duplicates before:", duplicates_before)
print("Rows with missing/edited values before:", missing_rows_before)

# Drop duplicate records so repeated rows do not bias the model.
df = df.drop_duplicates().reset_index(drop=True)

# Drop rows with missing values before dtype conversion.
df = df.dropna().reset_index(drop=True)

# Remove rows outside valid coding ranges and require integer survey codes except BMI.
valid_mask = pd.Series(True, index=df.index)
integer_columns = [col for col in EXPECTED_COLUMNS if col != "BMI"]
for column, (low, high) in VALUE_RANGES.items():
    in_range = df[column].between(low, high)
    integer_like = np.isclose(df[column], np.round(df[column])) if column in integer_columns else True
    valid_mask &= in_range & integer_like

invalid_rows = int((~valid_mask).sum())
df = df.loc[valid_mask].reset_index(drop=True)

# Convert binary/ordinal columns to int and keep BMI numeric.
df[integer_columns] = df[integer_columns].round().astype(int)
df["BMI"] = df["BMI"].astype(float)

print("Invalid/out-of-range rows removed:", invalid_rows)
print("Duplicates after:", int(df.duplicated().sum()))
print("Missing cells after:", int(df.isna().sum().sum()))
print("Clean shape:", df.shape)
df.dtypes


## 4. Exploratory Data Analysis

In [ ]:
# Plot the number of non-diabetic and diabetic records after cleaning.
plt.figure(figsize=(5.5, 3.8))
sns.countplot(data=df, x="Diabetes_binary", hue="Diabetes_binary", palette={0: TEAL, 1: AMBER}, legend=False)
plt.title("Class Balance (0 = No Diabetes, 1 = Diabetes)")
plt.xlabel("Diabetes_binary")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "01_class_balance.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Plot BMI distribution separately for each target class.
plt.figure(figsize=(7.5, 4.2))
sns.histplot(data=df, x="BMI", hue="Diabetes_binary", bins=40, kde=True, palette={0: TEAL, 1: AMBER})
plt.title("BMI Distribution by Diabetes Status")
plt.xlabel("BMI")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "02_bmi_distribution.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Use a boxplot to compare BMI spread for both diabetes classes.
plt.figure(figsize=(5.5, 4.0))
sns.boxplot(data=df, x="Diabetes_binary", y="BMI", hue="Diabetes_binary", palette={0: TEAL, 1: AMBER}, legend=False)
plt.title("BMI by Diabetes Status")
plt.xlabel("Diabetes_binary")
plt.ylabel("BMI")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "03_bmi_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# Compare diabetes counts for high blood pressure and high cholesterol.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for axis, column, title in zip(axes, ["HighBP", "HighChol"], ["High Blood Pressure", "High Cholesterol"]):
    sns.countplot(data=df, x=column, hue="Diabetes_binary", palette={0: TEAL, 1: AMBER}, ax=axis)
    axis.set_title(title)
    axis.set_xlabel(f"{column} (0 = No, 1 = Yes)")
    axis.legend(title="Diabetes", labels=["No", "Yes"])
plt.tight_layout()
plt.savefig(PLOTS_DIR / "04_bp_chol.png", dpi=200, bbox_inches="tight")
plt.show()


## 5. Correlation analysis

In [ ]:
# Compute pairwise correlations between all cleaned numeric columns.
corr = df.corr()

# Save a heatmap for the report and presentation.
plt.figure(figsize=(11, 8.5))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-0.5, vmax=0.5, square=True, cbar_kws={"shrink": 0.72})
plt.title("Correlation Matrix — BRFSS Health Indicators")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "05_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# List the ten features with the strongest absolute correlation with diabetes.
top_corr = corr["Diabetes_binary"].drop("Diabetes_binary").abs().sort_values(ascending=False).head(10)
print("Top 10 features most correlated with Diabetes:")
print(top_corr.round(3))


## 6. Train/test split

In [ ]:
# Separate the target column from the 21 health-indicator input features.
X = df.drop("Diabetes_binary", axis=1)
y = df["Diabetes_binary"]

# Use a stratified 80/20 split so both classes stay balanced in train and test sets.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)
print("Train:", X_train.shape, " Test:", X_test.shape)


## 7. Train Random Forest

In [ ]:
# Train a Random Forest because it handles mixed survey features well and gives feature importance.
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
model.fit(X_train, y_train)
print("Training done.")


## 8. Evaluate

In [ ]:
# Predict on the held-out test set only.
y_pred = model.predict(X_test)

# Calculate the main classification metrics.
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f} ({acc * 100:.2f}%)")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1 score : {f1:.4f}")

# Print confusion matrix and classification report for detailed evaluation.
cm = confusion_matrix(y_test, y_pred)
print("
Confusion matrix:")
print(pd.DataFrame(cm, index=["actual: No Diabetes", "actual: Diabetes"], columns=["pred: No Diabetes", "pred: Diabetes"]))
print("
Classification report:")
print(classification_report(y_test, y_pred, target_names=["No Diabetes", "Diabetes"]))


In [ ]:
# Save a clean confusion-matrix plot for the report and slides.
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ConfusionMatrixDisplay(cm, display_labels=["No Diabetes", "Diabetes"]).plot(ax=ax, cmap="GnBu", colorbar=False, values_format="d")
ax.set_title("Confusion Matrix — Random Forest")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "06_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()


## 9. Feature importance — top risk factors

In [ ]:
# Rank model features by Random Forest importance score.
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)

# Plot all feature-importance values horizontally.
plt.figure(figsize=(7.5, 7.2))
importance.plot(kind="barh", color=TEAL)
plt.title("Random Forest — Feature Importance")
plt.xlabel("Importance score")
plt.tight_layout()
plt.savefig(PLOTS_DIR / "07_feature_importance.png", dpi=200, bbox_inches="tight")
plt.show()

# Print the top five risk factors for the report and Streamlit sidebar.
top5 = importance.sort_values(ascending=False).head(5)
print("Top 5 diabetes risk factors:")
print(top5.round(4))


## 10. Save the model and metadata

In [ ]:
# Save the trained model with compression so it stays small enough for GitHub.
joblib.dump(model, MODELS_DIR / "rf_model.joblib", compress=3)

# Save the feature order; the Streamlit app uses this to avoid wrong-column predictions.
with open(MODELS_DIR / "feature_names.json", "w", encoding="utf-8") as file:
    json.dump(list(X.columns), file, indent=2)

# Save metrics for the Streamlit sidebar and project documentation.
with open(MODELS_DIR / "metrics.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "test_accuracy": round(float(acc), 4),
            "precision": round(float(prec), 4),
            "recall": round(float(rec), 4),
            "f1": round(float(f1), 4),
            "records_used": int(len(df)),
            "test_size": int(len(y_test)),
            "n_features": int(X.shape[1]),
            "top_5_features": top5.round(4).to_dict(),
        },
        file,
        indent=2,
    )

print("Saved model artifacts in:", MODELS_DIR)


## 11. Run the Streamlit app

Open a terminal at the **project root** and run:

```bash
streamlit run app/streamlit_app.py
```

If your terminal is currently inside `notebooks/`, first run `cd ..` to return to the project root.
